# অধ্যায় ৭: চ্যাট অ্যাপ্লিকেশন তৈরি করা
## Azure OpenAI API দ্রুত শুরু


## ওভারভিউ
এই নোটবুকটি [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) থেকে অনুকূলিত যা নোটবুকগুলো অন্তর্ভুক্ত করে যা [OpenAI](notebook-openai.ipynb) সেবাটিতে অ্যাক্সেস করে।

Python OpenAI API Azure OpenAI’র সাথেও কাজ করে, কিছু পরিবর্তনের সাথে। এখানে পার্থক্যগুলো সম্পর্কে আরও জানুন: [Python দিয়ে OpenAI এবং Azure OpenAI endpoints এর মধ্যে কীভাবে সুইচ করবেন](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

আরও দ্রুত শুরু উদাহরণগুলির জন্য অনুগ্রহ করে অফিসিয়াল [Azure OpenAI Quickstart Documentation](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) দেখুন


## বিষয়বস্তু সূচি  

[সংক্ষিপ্ত বিবরণ](#overview)  
[Azure OpenAI পরিষেবা দিয়ে শুরু করা](#getting-started-with-azure-openai-service)  
[আপনার প্রথম প্রম্পট তৈরি করুন](#build-your-first-prompt)  

[ব্যবহার ক্ষেত্রসমূহ](#use-cases)    
[1. পাঠ সংক্ষিপ্ত করুন](#summarize-text)  
[2. শ্রেণীবদ্ধকরণ পাঠ](#classify-text)  
[3. নতুন পণ্যের নাম তৈরি করুন](#generate-new-product-names)  
[4. একটি শ্রেণীবিভাজক সঠিক করুন](#fine-tune-a-classifier)  
[5. এমবেডিংস](#embeddings)

[তথ্যসূত্র](#references)


### Azure OpenAI পরিষেবার সাথে শুরু করা

নতুন গ্রাহকদের Azure OpenAI পরিষেবায় [অ্যাক্সেসের জন্য আবেদন করতে হবে](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst)।  
অনুমোদন প্রক্রিয়া সম্পন্ন হলে, গ্রাহকরা Azure পোর্টালে লগইন করতে পারেন, একটি Azure OpenAI পরিষেবা রিসোর্স তৈরি করতে পারেন, এবং স্টুডিওর মাধ্যমে মডেলগুলি পরীক্ষা শুরু করতে পারেন  

[দ্রুত শুরু করার জন্য চমৎকার সংস্থান](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### আপনার প্রথম প্রম্পট তৈরি করুন  
এই সংক্ষিপ্ত অনুশীলনটি একটি সাধারণ কাজ "সারাংশ তৈরির জন্য" একটি OpenAI মডেলে প্রম্পট জমা দেওয়ার মৌলিক পরিচয় প্রদান করবে।  


**ধাপসমূহ**:  
1. আপনার পাইথন পরিবেশে OpenAI লাইব্রেরি ইন্সটল করুন  
2. স্ট্যান্ডার্ড হেল্পার লাইব্রেরি লোড করুন এবং আপনি যে OpenAI সার্ভিস তৈরি করেছেন তার জন্য সাধারণ OpenAI সিকিউরিটি শংসাপত্র সেট করুন  
3. আপনার কাজের জন্য একটি মডেল বেছে নিন  
4. মডেলের জন্য একটি সাদাসিধে প্রম্পট তৈরি করুন  
5. মডেল API-তে আপনার অনুরোধ জমা দিন!  


### ১। OpenAI ইনস্টল করুন


  > [!NOTE] যদি এই নোটবুক কোডস্পেসে বা ডেভকন্টেইনারের মধ্যে চালানো হয় তবে এই ধাপটি প্রয়োজন হয় না


In [ ]:
%pip install openai python-dotenv

### ২. সহায়ক লাইব্রেরিগুলি ইমপোর্ট করুন এবং শংসাপত্রগুলি তৈরি করুন


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. সঠিক মডেল খোঁজা  
GPT-4o এবং GPT-4o মিনি-এর মতো মডেলগুলি প্রাকৃতিক ভাষা বুঝতে এবং তৈরি করতে পারে। Microsoft Foundry তে Azure OpenAI মডেল ক্ষমতার একটি পরিসর প্রদান করে, যাদের প্রত্যেকটির শক্তি এবং গতি ভিন্ন এবং বিভিন্ন কাজের জন্য উপযোগী। 

[Azure OpenAI in Microsoft Foundry models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## ৪. প্রম্পট ডিজাইন  

"বড় ভাষার মডেলগুলির জাদু হল যে বিশাল পরিমাণ টেক্সটে এই পূর্বানুমান ত্রুটি হ্রাস করার জন্য প্রশিক্ষিত হওয়ার মাধ্যমে, মডেলগুলি এমন ধারণা শেখে যা এই পূর্বানুমানের জন্য উপকারী। উদাহরণস্বরূপ, তারা নিম্নলিখিত ধারণাগুলো শেখে"(1):

* কিভাবে বানান করতে হয়
* কিভাবে ব্যাকরণ কাজ করে
* কিভাবে প্যারাফ্রেজ করতে হয়
* কিভাবে প্রশ্নের উত্তর দিতে হয়
* কিভাবে কথোপকথন চালাতে হয়
* কিভাবে বিভিন্ন ভাষায় লিখতে হয়
* কিভাবে কোড লেখা হয়
* ইত্যাদি।

#### কিভাবে একটি বড় ভাষার মডেল নিয়ন্ত্রণ করবেন  
"বড় ভাষার মডেলের সব ইনপুটের মধ্যে, সর্বাধিক প্রভাবশালী হল টেক্সট প্রম্পট(1)।

বড় ভাষার মডেলগুলোকে কয়েকটি উপায়ে প্রম্পট দেওয়া যায়:

- নির্দেশনা: মডেলকে বলুন আপনি কি চান
- সম্পূর্ণতা: মডেলকে আপনার যা চাওয়ার শুরু টুকুও পূরণ করতে বাধ্য করুন
- প্রদর্শনী: মডেলকে দেখান আপনি কি চান, যেভাবে:
- প্রম্পটে কয়েকটি উদাহরণ দিয়ে
- ফাইন-টিউনিং ট্রেনিং ডেটাসেটে কয়েকশো বা হাজার হাজার উদাহরণ দিয়ে



#### প্রম্পট তৈরি করার তিনটি মৌলিক নিয়মাবলী রয়েছে:

**দেখান এবং বলুন।** আপনি কি চান তা স্পষ্ট করুন, হয় নির্দেশনার মাধ্যমে, উদাহরণের মাধ্যমে, অথবা উভয়ের সংমিশ্রণে। যদি আপনি মডেলটিকে একটি তালিকা বর্ণানুক্রমিকভাবে সাজাতে বা একটি অনুচ্ছেদকে অনুভূতির ভিত্তিতে শ্রেণীবদ্ধ করতে চান, দেখান এটি আপনি যা চান।

**গুণগত তথ্য সরবরাহ করুন।** আপনি যদি একটি শ্রেণীবিন্যাসকারী তৈরি করতে চান বা মডেলটিকে একটি প্যাটার্ন অনুসরণ করাতে চান, তবে নিশ্চিত করুন যে পর্যাপ্ত উদাহরণ রয়েছে। আপনার উদাহরণগুলি প্রুফরিড করা নিশ্চিত করুন—মডেল সাধারণ বানান ভুল দেখতে সাধারণত যথেষ্ট বুদ্ধিমান এবং আপনাকে একটি উত্তর দিতে পারে, তবে এটি এই ভুলগুলি সচেতনভাবে করা হয়েছে ধরে নিতে পারে এবং এটি প্রতিক্রিয়াকে প্রভাবিত করতে পারে।

**আপনার সেটিংস পরীক্ষা করুন।** তাপমাত্রা এবং top_p সেটিংস কিভাবে মডেল কোনো উত্তর তৈরি করতে কতটা নির্ধারিত তা নিয়ন্ত্রণ করে। আপনি যদি এমন একটি উত্তর চান যেখানে কেবল একটিই সঠিক, তাহলে আপনি এগুলো কম সেট করতে চাইবেন। আপনি যদি আরও বৈচিত্র্যময় উত্তর চান, তবে এগুলো বেশি সেট করতে চাইবেন। এই সেটিংস নিয়ে মানুষদের সবচেয়ে বড় ভুল হল এগুলোকে "বুদ্ধিমত্তা" বা "সৃজনশীলতা" নিয়ন্ত্রণ হিসেবে ধরে নেওয়া।


উৎস: https://learn.microsoft.com/azure/ai-foundry/openai/overview


ইমেজ আপনার প্রথম টেক্সট প্রম্পট তৈরি করছে!


### ৫. জমা দিন!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### একই কলটিকে পুনরাবৃত্তি করুন, ফলাফলগুলি কিভাবে তুলনা করা যায়?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## টেক্সট সারাংশ  
#### চ্যালেঞ্জ  
একটি টেক্সট প্যাসেজের শেষে 'tl;dr:' যোগ করে টেক্সট সারাংশ করা। লক্ষ্য করুন কিভাবে মডেল অতিরিক্ত নির্দেশ ছাড়াই বিভিন্ন কাজ সম্পাদন করতে পারে। আপনি মডেলের আচরণ পরিবর্তন এবং প্রাপ্ত সারাংশকে কাস্টমাইজ করার জন্য tl;dr এর চেয়ে আরও বর্ণনামূলক প্রম্পট পরীক্ষা করতে পারেন(3)।  

সাম্প্রতিক কাজগুলি বড় পরিমাণে টেক্সট কর্পাসে প্রি-ট্রেনিং এবং নির্দিষ্ট কাজের জন্য ফাইন-টিউনিং করে অনেক NLP কাজ এবং বেঞ্চমার্কে বিশাল উন্নতি প্রদর্শন করেছে। যদিও সাধারণত আর্কিটেকচারে টাস্ক-অ্যাগনস্টিক হলেও, এই পদ্ধতিটি এখনও হাজার হাজার বা তিরিশ হাজার উদাহরণের টাস্ক-নির্দিষ্ট ফাইন-টিউনিং ডেটাসেটের প্রয়োজন। তুলনায়, মানুষ সাধারণত মাত্র কয়েকটি উদাহরণ বা সাধারণ নির্দেশ থেকে একটি নতুন ভাষা কাজ করতে পারে - যা বর্তমান NLP সিস্টেমগুলি এখনও মূলত করতে পারে না। এখানে আমরা দেখাই যে ভাষা মডেল স্কেল আপ করলে টাস্ক-অ্যাগনস্টিক, কয়েক-শট পারফরম্যান্স অনেক উন্নত হয়, কখনও কখনও আগের সর্বোত্তম ফাইন-টিউনিং পদ্ধতির সাথে প্রতিযোগিতা করতে পারে।  



সারাংশ  


# একাধিক ব্যবহারের জন্য অনুশীলন  
1. পাঠ সংক্ষেপণ  
2. পাঠ শ্রেণীবদ্ধকরণ  
3. নতুন পণ্য নাম তৈরি করুন
4. এম্বেডিংস
5. একটি শ্রেণীবদ্ধকারী ফাইন-টিউন করা


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## টেক্সট শ্রেণীবদ্ধকরণ  
#### চ্যালেঞ্জ  
ইনফারেন্স সময় প্রদত্ত শ্রেণীগুলিতে আইটেমগুলি শ্রেণীবদ্ধ করুন। নিম্নলিখিত উদাহরণে, আমরা প্রম্পটে (*playground_reference) উভয় শিরোনাম এবং শ্রেণীবদ্ধ করার জন্য টেক্সট প্রদান করি। 

গ্রাহক অনুসন্ধান: হ্যালো, সম্প্রতি আমার ল্যাপটপ কীবোর্ডের একটি কী ভেঙে গেছে এবং আমার একটি বিকল্প দরকার হবে:

শ্রেণীবদ্ধকৃত শিরোনাম:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## নতুন পণ্যের নাম তৈরি করুন
#### চ্যালেঞ্জ
উদাহরণ শব্দ থেকে পণ্যের নাম তৈরি করুন। এখানে আমরা এমন একটি পণ্যের সম্পর্কে তথ্য প্রদান করেছি যার জন্য আমরা নাম তৈরি করতে যাচ্ছি। আমরা একটি অনুরূপ উদাহরণও দিয়েছি যা আমরা পেতে চাই এমন ধরণ দেখানোর জন্য। আমরা র‍্যান্ডমনেস বাড়াতে এবং আরও উদ্ভাবনী উত্তর পেতে তাপমাত্রার মানও বেশি করেছি।

পণ্যের বিবরণ: একটি বাড়ির মিল্কশেক মেকার
বীজ শব্দ: দ্রুত, স্বাস্থ্যকর, কমপ্যাক্ট।
পণ্যের নাম: HomeShaker, Fit Shaker, QuickShake, Shake Maker

পণ্যের বিবরণ: এমন একটি জোড়া জুতা যা যেকোনো পায়ের আকারে মানানসই হতে পারে।
বীজ শব্দ: অভিযোজ্য, মানানসই, অলমনি-ফিট।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## এমবেডিংস  
এই বিভাগে এমবেডিংস কীভাবে উদ্ধার করতে হয় এবং শব্দ, বাক্য, এবং ডকুমেন্টের মধ্যে মিল কীভাবে খুঁজে বের করতে হয় তা দেখানো হবে। নিম্নোক্ত নোটবুকগুলি চালানোর জন্য, আপনাকে একটি মডেল ডিপ্লয় করতে হবে যা `text-embedding-ada-002` কে বেস মডেল হিসেবে ব্যবহার করে এবং তার ডিপ্লয়মেন্ট নামটি .env ফাইলে `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` ভেরিয়েবল ব্যবহার করে সেট করতে হবে।  


### মডেল শ্রেণীবিন্যাস - একটি এমবেডিং মডেল বাছাই করা

**মডেল শ্রেণীবিন্যাস**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (এমবেডিংস পরিবার)  
{capability} --> ada             (সমস্ত অন্যান্য এমবেডিং মডেল ২০২৪ সালে অবসান হবে)  
{input-type} --> n/a             (শুধুমাত্র সার্চ মডেলের ক্ষেত্রে নির্দিষ্ট)  
{identifier} --> 002             (ভার্সন ০০২)  

model = 'text-embedding-ada-002'


  > [!NOTE] Codespaces-এ বা Devcontainer-এর ভিতরে এই নোটবুক চালানো হলে নিচের ধাপটি প্রয়োজন নেই


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## CNN দৈনিক খবর ডেটাসেট থেকে নিবন্ধ তুলনা করা
উৎস: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# রেফারেন্স  
- [Microsoft Foundry - Azure OpenAI Models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# আরও সাহায্যের জন্য  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# সহায়করা
* লুইস লি  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
